# NB09 · Molecular Fingerprints: Keys, Paths, Circles, and Pharmacophore Pairs

**Part C: Representations & Descriptors** · Estimated time: ~90 min · Difficulty: ●●○

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prateekpaul/Cheminformatics/blob/main/notebooks/nb09_fingerprints_keys_paths_circles.ipynb)

---

### One-Line Summary
Master the modern RDKit `rdFingerprintGenerator` API, explore the four major fingerprint paradigms (substructure keys, path-based, circular ECFP/FCFP, and topological distance/pharmacophore pairs), investigate folding length and bit collisions, and decode bit environments using `bitInfo` to construct visual substructure galleries.

## 1. Learning Objectives

By the end of this notebook, you will be able to:
1. **Apply the Modern API**: Utilize `rdFingerprintGenerator` instead of deprecated legacy calls (addressing Course Errata E3).
2. **Distinguish the 4 Fingerprint Families**: Contrast Substructure Keys (MACCS), Path-based (Daylight/RDKit/FP2), Circular (Morgan/ECFP4/ECFP6), and Distance-based (Atom Pairs & Topological Torsions).
3. **Clarify ECFP Radius vs. Diameter**: Understand why radius 2 corresponds to diameter 4 (ECFP4) and radius 3 corresponds to diameter 6 (ECFP6).
4. **Construct 2D Pharmacophore Signatures**: Explain the combinatorial distance-binning of `Pharm2D` Gobbi signatures and understand feature point geometry.
5. **Analyze Hashing, Folding & Bit Collisions**: Quantify bit saturation and Tanimoto similarity distortion when folding fingerprints from 4096 down to 128 bits.
6. **Map Bits to Substructures**: Extract `bitInfo` from Morgan fingerprints and render a visual bit-to-substructure gallery explaining which chemical fragments activate individual bits.

## 2. Prerequisites & Lecture Links

- **Prerequisites**: SMILES strings (`nb01`, `nb03`), molecular graphs (`nb04`), and functional groups (`nb07`).
- **Lecture Aligned**: Lecture Deck 06 (*Similarity Search & Fingerprints*), Lecture Deck 04 (*Organic Structures*), and Lecture Deck 03 (*Cheminformatics Tools*).
- **Errata Covered**:
  - **E3**: Deprecation of `GetMorganFingerprintAsBitVect`; corrected radius terminology (radius 2 = ECFP4, radius 3 = ECFP6).
  - **E4**: Open Babel FP2 bit length clarification (1,024 bits vs 256).

In [ ]:
# Setup and environment configuration
import os, sys, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rdkit import Chem, RDLogger, DataStructs
from rdkit.Chem import AllChem, Draw, MACCSkeys, rdFingerprintGenerator
from rdkit.Chem.Draw import rdMolDraw2D

# Ensure src directory is in sys.path
repo_root = Path.cwd().parent if Path.cwd().name in ["notebooks", "solutions"] else Path.cwd()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import chemlib
from chemlib import fingerprints

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

sns.set_theme(style="whitegrid", font_scale=1.1)
print("Environment initialized successfully. chemlib version:", chemlib.__version__)
# Course reproducibility seeds and execution mode
import random
FULL_RUN = False


## 3. The Molecular Fingerprint Paradigm

A **molecular fingerprint** is a fixed-length or hashed bit vector (or count vector) representing the chemical features of a molecule. Fingerprints enable ultra-fast boolean operations (bitwise `AND`, `OR`, `POPCNT`), allowing millions of pairwise similarity comparisons per second.

### Fingerprint Taxonomy Table

| Family | Algorithm / Format | Typical Bit Length | Hashed? | Reversible to Pattern? | Primary Application |
|:---|:---|:---|:---|:---|:---|
| **Substructure Keys** | **MACCS Keys** | 166 bits | No | Yes (predefined SMARTS dictionary) | Coarse screening, chemical diversity profiling |
| **Substructure Keys** | **PubChem Keys** | 881 bits | No | Yes (explicit dictionary) | Broad public screening, biological activity clustering |
| **Path-Based** | **RDKit Topological** | 2048 bits | Yes | No (collisions occur) | Linear and branched substructure matching |
| **Path-Based** | **Open Babel FP2** | 1024 bits | Yes | No | Fast pre-filtering (linear paths 1–7 atoms) |
| **Circular (ECFP4)** | **Morgan (radius 2)** | 2048 bits | Yes | No (can map to atom env via `bitInfo`) | Gold standard for ligand-based virtual screening & QSAR |
| **Circular (ECFP6)** | **Morgan (radius 3)** | 2048 bits | Yes | No | Larger macrocyclic / natural product environments |
| **Distance-Based** | **Atom Pairs** | 2048 bits | Yes | Partial (atom types + path distance) | Scaffold hopping, distant analog retrieval |
| **Distance-Based** | **Topological Torsions** | 2048 bits | Yes | Partial (4-atom bonded sequences) | Conformationally relevant local geometry |
| **Pharmacophore** | **2D Pharm2D (Gobbi)** | 39,973 bits | Combinatorial | Yes (distance bins between pharmacophoric points) | Receptor-independent pharmacophore matching |

## 4. The Modern `rdFingerprintGenerator` API & Errata E3

> [!IMPORTANT]
> **Lecture Errata (E3) & Modern Best Practice**:
> In historical slides and older RDKit tutorials, circular fingerprints were generated via:
> `AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=2048)`
> **This API is deprecated.** Modern RDKit code uses the unified `rdFingerprintGenerator` framework, which provides reusable, thread-safe generator objects with consistent configuration.
>
> **Radius vs. Diameter Terminology**:
> - **Radius 1** = Diameter 2 = **ECFP2**
> - **Radius 2** = Diameter 4 = **ECFP4** (the standard)
> - **Radius 3** = Diameter 6 = **ECFP6**
> *Never set radius=3 when ECFP4 is intended!*

In [ ]:
# Instantiate modern fingerprint generators
fpgen_maccs = MACCSkeys.GenMACCSKeys                                                    # Substructure keys (166)
fpgen_rdkit = rdFingerprintGenerator.GetRDKitFPGenerator(minPath=1, maxPath=7, fpSize=2048) # Path-based
fpgen_ecfp4 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)          # Circular ECFP4
fpgen_ecfp6 = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=2048)          # Circular ECFP6
fpgen_atompair = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=2048)              # Atom Pairs
fpgen_torsion = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=2048)    # Topological Torsions

print("Fingerprint generators constructed successfully.")

In [ ]:
# Compare fingerprint bit density (number of ON bits) on Aspirin
aspirin = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")

fp_maccs = MACCSkeys.GenMACCSKeys(aspirin)
fp_rdkit = fpgen_rdkit.GetFingerprint(aspirin)
fp_ecfp4 = fpgen_ecfp4.GetFingerprint(aspirin)
fp_ecfp6 = fpgen_ecfp6.GetFingerprint(aspirin)
fp_ap = fpgen_atompair.GetFingerprint(aspirin)
fp_tt = fpgen_torsion.GetFingerprint(aspirin)

fp_stats = [
    ("MACCS Keys", fp_maccs.GetNumBits(), fp_maccs.GetNumOnBits()),
    ("RDKit Topological", fp_rdkit.GetNumBits(), fp_rdkit.GetNumOnBits()),
    ("Morgan ECFP4 (radius=2)", fp_ecfp4.GetNumBits(), fp_ecfp4.GetNumOnBits()),
    ("Morgan ECFP6 (radius=3)", fp_ecfp6.GetNumBits(), fp_ecfp6.GetNumOnBits()),
    ("Atom Pairs", fp_ap.GetNumBits(), fp_ap.GetNumOnBits()),
    ("Topological Torsions", fp_tt.GetNumBits(), fp_tt.GetNumOnBits()),
]

df_stats = pd.DataFrame(fp_stats, columns=["Fingerprint_Type", "Total_Bits", "ON_Bits"])
df_stats["Bit_Density_%"] = (df_stats["ON_Bits"] / df_stats["Total_Bits"]) * 100.0
print("Aspirin Fingerprint Profile:")
df_stats

## 5. 2D Pharmacophore Fingerprints: Gobbi Signatures

Pharmacophore fingerprints represent molecules not by atomic connectivity, but by spatial or topological relationships between **pharmacophoric feature points**:
- **H-Bond Acceptor (HBA)**
- **H-Bond Donor (HBD)**
- **Positive Ionizable / Basic (PI)**
- **Negative Ionizable / Acidic (NI)**
- **Aromatic / Hydrophobic (AR)**

### Combinatorial Arithmetic
In a **3-point pharmacophore** (triplet of features), each feature pair is separated by a topological distance binned into discrete intervals (e.g., 2–3 bonds, 4–5 bonds, 6–7 bonds, etc.).
- A small toy system with 2 feature patterns, 2–3 point triangles, and 2 distance bins produces $6 + 32 = 38$ bits.
- The full **Gobbi `Pharm2D` signature** implemented in RDKit contains **39,973 bits**, capturing all 2-point and 3-point pharmacophoric triangles across 8 feature classes and 7 distance bins!

In [ ]:
# Compute Gobbi 2D Pharmacophore fingerprint for Aspirin and Caffeine
from rdkit.Chem.Pharm2D import Generate, Gobbi_Pharm2D

caffeine = Chem.MolFromSmiles("Cn1cnc2c1c(=O)n(C)c(=O)n2C")

pharm_asp = Generate.Gen2DFingerprint(aspirin, Gobbi_Pharm2D.factory)
pharm_caff = Generate.Gen2DFingerprint(caffeine, Gobbi_Pharm2D.factory)

print(f"Gobbi Pharm2D Signature total bit size: {pharm_asp.GetNumBits()}")
print(f"Aspirin pharmacophore ON bits: {pharm_asp.GetNumOnBits()}")
print(f"Caffeine pharmacophore ON bits: {pharm_caff.GetNumOnBits()}")

sim_pharm = DataStructs.TanimotoSimilarity(pharm_asp, pharm_caff)
print(f"Pharm2D Tanimoto Similarity between Aspirin and Caffeine: {sim_pharm:.4f}")

## 6. Hashing, Folding, and Bit Collision Analysis

When circular substructure environments are generated, they are assigned a 32-bit integer hash code ($0$ to $2^{32}-1$). Because storing $4.3 \times 10^9$ bits per molecule is memory-prohibitive, the integer is **folded** into a fixed-length bit vector of length $L$ via modulo division:
$$\text{bit\_index} = \text{hash\_code} \pmod L$$

### The Bit Collision Phenomenon
If two chemically distinct fragments happen to satisfy $\text{hash}_1 \equiv \text{hash}_2 \pmod L$, they set the **exact same bit position**.
- At large lengths ($L = 4096$ or $2048$), collisions are rare.
- At small lengths ($L \le 256$), bit collisions multiply, causing **saturation** (too many bits ON) and artificially **inflating Tanimoto similarity** between unrelated molecules!

In [ ]:
# Folding experiment: evaluate bit density and similarity distortion from 4096 down to 128 bits
salicylic_acid = Chem.MolFromSmiles("Oc1ccccc1C(=O)O") # structurally related analog

bit_sizes = [4096, 2048, 1024, 512, 256, 128]
folding_results = []

for n_bits in bit_sizes:
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
    fp_a = gen.GetFingerprint(aspirin)
    fp_s = gen.GetFingerprint(salicylic_acid)
    
    tan = DataStructs.TanimotoSimilarity(fp_a, fp_s)
    density = (fp_a.GetNumOnBits() / n_bits) * 100.0
    folding_results.append({
        "Bit_Length": n_bits,
        "Aspirin_ON_Bits": fp_a.GetNumOnBits(),
        "Bit_Density_%": round(density, 2),
        "Tanimoto_Similarity": round(tan, 4)
    })

df_folding = pd.DataFrame(folding_results)
print("Impact of Folding Length on Bit Density & Tanimoto Similarity:")
df_folding

In [ ]:
# Plot bit density and similarity as a function of bit size
fig, ax1 = plt.subplots(figsize=(7.5, 4.5))

color = '#1f77b4'
ax1.set_xlabel('Fingerprint Length (Bits)', fontweight='bold')
ax1.set_ylabel('Bit Density (%)', color=color, fontweight='bold')
ax1.plot(df_folding['Bit_Length'], df_folding['Bit_Density_%'], color=color, marker='o', lw=2)
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_xscale('log', base=2)

ax2 = ax1.twinx()
color = '#d62728'
ax2.set_ylabel('Tanimoto Similarity (Aspirin vs Salicylic Acid)', color=color, fontweight='bold')
ax2.plot(df_folding['Bit_Length'], df_folding['Tanimoto_Similarity'], color=color, marker='s', lw=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Effect of Fingerprint Folding: Saturation & Similarity Inflation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Explainable AI: Mapping Bits to Atomic Environments (`bitInfo`)

A frequent criticism of circular fingerprints in medicinal chemistry is that they are "black boxes". However, RDKit allows full inspection of the exact atomic substructure responsible for setting any bit via the **`bitInfo` map**:

$$\text{bit\_id} \longrightarrow [(\text{atom\_idx}_1, \text{radius}_1), (\text{atom\_idx}_2, \text{radius}_2), \dots]$$

- **Radius 0**: Single atom with its atomic number, degree, and charge (e.g. carbonyl oxygen).
- **Radius 1**: Central atom plus its immediate bonded neighbors (e.g. ester carbonyl $O=C-O$).
- **Radius 2**: Central atom, immediate neighbors, and next-nearest neighbors (diameter = 4 bonds).

In [ ]:
# Extract bitInfo using AdditionalOutput in modern RDKit
ao = rdFingerprintGenerator.AdditionalOutput()
ao.AllocateBitInfoMap()

fp_aspirin_info = fpgen_ecfp4.GetFingerprint(aspirin, additionalOutput=ao)
bit_info = ao.GetBitInfoMap()

print(f"Total unique bits set in Aspirin: {len(bit_info)}")
# Display the first 5 bits and their atom environments
for bit_id in list(bit_info.keys())[:5]:
    atom_radius_pairs = bit_info[bit_id]
    print(f"  Bit {bit_id:4d} -> Set by {len(atom_radius_pairs)} center(s): {atom_radius_pairs}")

In [ ]:
# Render the substructure environment for the first 3 bits in Aspirin
sample_bits = list(bit_info.keys())[:3]
print("Visualizing 3 sample Morgan bit environments in Aspirin:")
for b in sample_bits:
    center_atom, radius = bit_info[b][0]
    print(f"Bit {b}: Central atom = {aspirin.GetAtomWithIdx(center_atom).GetSymbol()} (idx {center_atom}), Radius = {radius}")

# Draw the bits
img_bits = Draw.DrawMorganBits(
    [(aspirin, b, bit_info) for b in sample_bits],
    molsPerRow=3,
    subImgSize=(220, 180),
    legends=[f"Bit {b} (rad {bit_info[b][0][1]})" for b in sample_bits]
)
img_bits

## 8. Hands-On Exercises

### Exercise 9.1: Multi-Fingerprint Similarity Benchmark
Write a function `compute_cross_similarities(mol1: Chem.Mol, mol2: Chem.Mol) -> dict` that computes the Tanimoto similarity between `mol1` and `mol2` across 4 different fingerprint types:
1. MACCS Keys
2. RDKit Topological (2048 bits)
3. Morgan ECFP4 (radius=2, 2048 bits)
4. Atom Pairs (2048 bits)

In [ ]:
def compute_cross_similarities(mol1: Chem.Mol, mol2: Chem.Mol) -> dict:
    # Complete working solution
    # 1. MACCS
    fp1_maccs = MACCSkeys.GenMACCSKeys(mol1)
    fp2_maccs = MACCSkeys.GenMACCSKeys(mol2)
    tan_maccs = DataStructs.TanimotoSimilarity(fp1_maccs, fp2_maccs)
    
    # 2. RDKit topological
    fp1_rdkit = fpgen_rdkit.GetFingerprint(mol1)
    fp2_rdkit = fpgen_rdkit.GetFingerprint(mol2)
    tan_rdkit = DataStructs.TanimotoSimilarity(fp1_rdkit, fp2_rdkit)
    
    # 3. ECFP4
    fp1_ecfp4 = fpgen_ecfp4.GetFingerprint(mol1)
    fp2_ecfp4 = fpgen_ecfp4.GetFingerprint(mol2)
    tan_ecfp4 = DataStructs.TanimotoSimilarity(fp1_ecfp4, fp2_ecfp4)
    
    # 4. Atom pairs
    fp1_ap = fpgen_atompair.GetFingerprint(mol1)
    fp2_ap = fpgen_atompair.GetFingerprint(mol2)
    tan_ap = DataStructs.TanimotoSimilarity(fp1_ap, fp2_ap)
    
    return {
        'MACCS': float(tan_maccs),
        'RDKit': float(tan_rdkit),
        'ECFP4': float(tan_ecfp4),
        'AtomPair': float(tan_ap)
    }

sims_demo = compute_cross_similarities(aspirin, caffeine)
print('Cross-fingerprint similarities between Aspirin and Caffeine:', sims_demo)

In [ ]:
# Self-check test for Exercise 9.1
sims_res = compute_cross_similarities(aspirin, caffeine)
assert "MACCS" in sims_res and "RDKit" in sims_res
assert "ECFP4" in sims_res and "AtomPair" in sims_res
for k, v in sims_res.items():
    assert 0.0 <= v <= 1.0, f"Similarity {k} must be in [0, 1], got {v}"
# Aspirin and caffeine are chemically distinct; ECFP4 should be low
assert sims_res["ECFP4"] < 0.25, f"Expected low ECFP4 similarity, got {sims_res['ECFP4']}"
print("✔ Exercise 9.1 passed!")

### Exercise 9.2: Substructure Bit Extraction
Using `bit_info` for Aspirin, write code to find all bits that have a circular radius of exactly $0$ (individual heavy atoms) and count how many unique radius-0 bits exist.

In [ ]:
# Complete working solution
radius_0_bits = [bit_id for bit_id, envs in bit_info.items() if any(r == 0 for atom_idx, r in envs)]
n_radius_0 = len(radius_0_bits)
print(f'Found {n_radius_0} unique radius-0 bits in Aspirin: {radius_0_bits}')

In [ ]:
# Self-check test for Exercise 9.2
assert n_radius_0 > 0, "Must find at least one radius-0 bit"
# Aspirin heavy atoms: 9 carbons (aliphatic, aromatic, carbonyl) + 4 oxygens -> typically 5-7 distinct atom types
assert 4 <= n_radius_0 <= 10, f"Expected 4 to 10 distinct radius-0 bits, got {n_radius_0}"
print("✔ Exercise 9.2 passed!")

### Exercise 9.3: Folding & Collision Drift
Compute the Tanimoto similarity between Aspirin and Salicylic acid using an ECFP4 generator folded to **64 bits**. Verify that due to severe bit collisions, the 64-bit Tanimoto similarity is significantly greater than the 2048-bit Tanimoto similarity.

In [ ]:
# Complete working solution
gen_64 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=64)
fp_asp_64 = gen_64.GetFingerprint(aspirin)
fp_sal_64 = gen_64.GetFingerprint(salicylic_acid)
tan_64 = float(DataStructs.TanimotoSimilarity(fp_asp_64, fp_sal_64))

gen_2048 = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp_asp_2048 = gen_2048.GetFingerprint(aspirin)
fp_sal_2048 = gen_2048.GetFingerprint(salicylic_acid)
tan_2048 = float(DataStructs.TanimotoSimilarity(fp_asp_2048, fp_sal_2048))

print(f'Tanimoto at 2048 bits: {tan_2048:.4f}')
print(f'Tanimoto at 64 bits:   {tan_64:.4f}')

In [ ]:
# Self-check test for Exercise 9.3
assert tan_64 > tan_2048, f"Expected collision-induced inflation (tan_64 > tan_2048), got {tan_64} vs {tan_2048}"
print("✔ Exercise 9.3 passed!")

## 9. Common Pitfalls

1. **ECFP Radius / Diameter Confusion**: Specifying `radius=4` in `GetMorganGenerator` generates **ECFP8** (an 8-bond diameter neighborhood), which is almost universally overfitted and sparse. Standard ECFP4 requires `radius=2`.
2. **Comparing Dissimilar Fingerprint Lengths**: Computing Tanimoto similarity between a 1024-bit fingerprint and a 2048-bit fingerprint will throw a dimension mismatch error in C++.
3. **Using Bit Vectors for Repeated Features**: In bit vectors, having 1 hydroxyl group vs 5 hydroxyl groups sets the exact same bit (bit is ON). When stoichiometry matters, use count fingerprints (`GetCountFingerprint`).

## 10. Summary & Key Takeaways

- **Modern RDKit**: Always use `rdFingerprintGenerator` for circular, path-based, and atom-pair fingerprints.
- **Four Paradigms**: Substructure keys (fixed keys), Path-based (linear subgraphs), Circular (radial Morgan environments), and Distance-based (atom pairs/pharmacophores).
- **Bit Collisions & Folding**: Folding compresses memory at the cost of bit collisions; lengths below 512 bits suffer from severe Tanimoto inflation.
- **Explainability**: `bitInfo` maps Morgan bits directly back to atom indices and radii, converting abstract bit vectors into interpretable visual substructures.

## 11. References & Further Reading

- Rogers, D.; Hahn, M. Extended-connectivity fingerprints. *J. Chem. Inf. Model.* **2010**, 50, 742–754.
- Durant, J. L. et al. Reoptimization of MDL keys for use in drug discovery. *J. Chem. Inf. Comput. Sci.* **2002**, 42, 1273–1280.
- Gobbi, A.; Lee, M. L. PHRP: Pharmacophore fingerprinting of small molecules. *J. Chem. Inf. Comput. Sci.* **2003**, 43, 317–323.
- Carhart, R. E. et al. Atom pairs as molecular features in structure-activity studies. *J. Chem. Inf. Comput. Sci.* **1985**, 25, 64–73.

In [ ]:
# Final self-test cell
print("✔ NB09 checks passed")